# Test MURED Dataloader

This notebook verifies the dataset loader for the
**MURED** dataset.

In [ ]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from config.constants import MURED_PATH
from dataloader.mured import MUREDDataset, MUREDModule, compute_mured_class_weights
from utils.transforms import eval_transform

print("Python path:", os.getcwd())
root = os.environ.get("MURED_PATH", MURED_PATH)
print("Using root:", root)

# label_col="DR" -> binary diabetic-retinopathy classification (0 = no DR, 1 = DR)
transform = eval_transform(224)
dm = MUREDModule(root=root, transform=transform, batch_size=8, label_col="DR")
dm.setup(stage="fit")
print("Dataset ready")

loader = dm.val_dataloader()
print("len(val_ds)=", len(dm.val_ds))

batch = next(iter(loader))
imgs, labels, paths = batch
print("batch images shape:", tuple(imgs.shape))
print("batch labels shape:", tuple(labels.shape), "dtype:", labels.dtype)
print("sample labels (0=no DR, 1=DR):", labels[:8].tolist())
print("sample paths:", list(paths[:3]))

# Binary class weights for DR
w = compute_mured_class_weights(root, label_col="DR")
print("DR class weights (class 0, class 1):", w.tolist())

/home/andremitri/miniconda3/envs/mae/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python path: /home/andremitri/code/LEMUR/notebooks
Using root: /exp/andremitri/mamba/aptos2019/versions/3
[APTOS] Split: train | Images: 2930
[APTOS] Split: val   | Images: 366
Dataset ready
len(val_ds)= 366
batch images shape: torch.Size([8, 3, 224, 224])
batch labels shape: torch.Size([8])
sample paths: ['/exp/andremitri/mamba/aptos2019/versions/3/val_images/val_images/000c1434d8d7.png', '/exp/andremitri/mamba/aptos2019/versions/3/val_images/val_images/001639a390f0.png', '/exp/andremitri/mamba/aptos2019/versions/3/val_images/val_images/0024cdab0c1e.png']


In [ ]:
import pandas as pd

# Inspect the DR column of the MURED CSVs
csv_path = os.path.join(root, "train_data.csv")
df = pd.read_csv(csv_path)
print("columns:", list(df.columns))
print("num rows:", len(df))
dr = df["DR"].astype(int)
print("DR negative (no DR):", int((dr == 0).sum()))
print("DR positive (DR):    ", int((dr == 1).sum()))
df["DR"].value_counts()

array([0, 1])

In [ ]:
# DR class balance across the available CSV splits
for f in ["train_data.csv", "test_data.csv"]:
    d = pd.read_csv(os.path.join(root, f))
    dr = d["DR"].astype(int)
    print(f"{f}: total={len(d)}  DR pos={int(dr.sum())}  DR neg={int((dr == 0).sum())}")